In [9]:
import time
import re
import random
import logging
import datetime
import argparse
import ast
import pandas as pd
import numpy as np
from huggingface_hub import login
from collections import Counter


In [17]:
full_train_df = pd.read_csv("/cluster/work/projects/ec403/ec-michechi/Project_M/data/landmark_evo_train.csv", na_values=['', 'None', 'NaN', 'na', 'nan']).fillna('')

/localscratch/1673041/ipykernel_3186741/3015138929.py:1: DtypeWarning: Columns (16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  full_train_df = pd.read_csv("/cluster/work/projects/ec403/ec-michechi/Project_M/data/landmark_evo_train.csv", na_values=['', 'None', 'NaN', 'na', 'nan']).fillna('')


In [21]:
filtered_datasets = []
for dataset in [full_train_df]:
    visit_counts = dataset['subject_id'].value_counts()
    selected_patients = visit_counts[visit_counts == 3].index
    dataset_selected = dataset[dataset['subject_id'].isin(selected_patients)].copy()
    filtered_datasets.append(dataset_selected)

train_df_selected = filtered_datasets[0]

In [23]:
train_df_selected.head(3)

,subject_id,hadm_id,admission_category,landmark_visit,age_at_landmark,gender,num_total_visits,death_in_90days,med_text,diag_text,...,no_more_diagnoses,no_more_medications,no_more_procedures,no_more_dose,meds_per_visit,diag_per_visit,proc_per_visit,dose_per_visit,gender_numeric,days_since_previous_visit
152303,10000826,20032235,EMERGENCY,1,32,F,1,0,Morphine Sulfate\nLorazepam\nHeparin\nDocusate...,"Alcoholic cirrhosis of liver\nPneumonia, organ...",...,,,,,"{1: ['Morphine Sulfate', 'Lorazepam', 'Heparin...","{1: ['Alcoholic cirrhosis of liver', 'Pneumoni...",{1: ['Percutaneous abdominal drainage']},{1: ['2-4 mg of Morphine Sulfate in SYR throug...,0,-1.0
152304,10000826,21086876,EMERGENCY,2,32,F,2,0,Morphine Sulfate\nLorazepam\nHeparin\nDocusate...,"Alcoholic cirrhosis of liver\nPneumonia, organ...",...,"Alcoholic cirrhosis of liver\nAnxiety state, u...",1/2 NS\nAcetaminophen\nCiprofloxacin HCl\nDocu...,,0.5 mg of Lorazepam in TAB through PO/NG\n0.5 ...,"{1: ['Morphine Sulfate', 'Lorazepam', 'Heparin...","{1: ['Alcoholic cirrhosis of liver', 'Pneumoni...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['2-4 mg of Morphine Sulfate in SYR throug...,0,6.0
152305,10000826,28289260,EMERGENCY,3,32,F,3,0,Morphine Sulfate\nLorazepam\nHeparin\nDocusate...,"Alcoholic cirrhosis of liver\nPneumonia, organ...",...,Alcoholic fatty liver\nOther and unspecified a...,0.9% Sodium Chloride\n5% Dextrose\nAcetylcyste...,,1 Appl of Sarna Lotion in BTL through TP\n1 PT...,"{1: ['Morphine Sulfate', 'Lorazepam', 'Heparin...","{1: ['Alcoholic cirrhosis of liver', 'Pneumoni...","{1: ['Percutaneous abdominal drainage'], 2: ['...",{1: ['2-4 mg of Morphine Sulfate in SYR throug...,0,6.0


In [24]:
def compact_narrative_prompt(row):
    narrative = f"You are a Doctor.\nWhat is the probability of death in the next 90 days for {row['age_at_landmark']}-year-old {row['gender']} patient?\n"
    current_visit = row['landmark_visit']
    type = row['admission_category']
    narrative += f"Visit number {current_visit} - {type} \n"

    if row['days_since_previous_visit'] != -1:
        narrative += f"Last visit happened {row['days_since_previous_visit']} days ago."

    # --- Diagnosi ---
    narrative += "\nDIAGNOSIS HISTORY:"
    diag_per_visit = ast.literal_eval(row['diag_per_visit'])
    
    # Conta frequenze
    all_diags = []
    for diags in diag_per_visit.values():
        all_diags.extend(diags)
    diag_counts = Counter(all_diags)

    # Diagnosi croniche (almeno 2 visite)
    chronic_diags = [d for d, c in diag_counts.items() if c >= 2]

    # Diagnosi nuove solo in questa visita
    current_diags = diag_per_visit[int(current_visit)]
    new_diags = [d for d in current_diags if diag_counts[d] == 1]

    if chronic_diags:
        narrative += f"\nChronic diagnoses: {', '.join(chronic_diags)}."
    if new_diags:
        narrative += f"\nNew diagnoses in this visit: {', '.join(new_diags)}."
    if not chronic_diags and not new_diags:
        narrative += "\nNo diagnoses recorded."

    # --- Farmaci ---
    narrative += "\nPRESCRIPTIONS HISTORY:"
    meds_per_visit = ast.literal_eval(row['meds_per_visit'])
    
    all_meds = []
    for meds in meds_per_visit.values():
        all_meds.extend(meds)
    med_counts = Counter(all_meds)

    chronic_meds = [m for m, c in med_counts.items() if c >= 2]
    current_meds = meds_per_visit[int(current_visit)]
    new_meds = [m for m in current_meds if med_counts[m] == 1]

    if chronic_meds:
        narrative += f"\nChronic medications: {', '.join(chronic_meds)}."
    if new_meds:
        narrative += f"\nNew medications in this visit: {', '.join(new_meds)}."
    if not chronic_meds and not new_meds:
        narrative += "\nNo medications recorded."

    # --- Procedure ---
    narrative += "\nPROCEDURES HISTORY:"
    proc_per_visit = ast.literal_eval(row['proc_per_visit'])

    all_proc = []
    for procs in proc_per_visit.values():
        all_proc.extend(procs)
    proc_counts = Counter(all_proc)

    chronic_proc = [p for p, c in proc_counts.items() if c >= 2]
    current_proc = proc_per_visit[int(current_visit)]
    new_proc = [p for p in current_proc if proc_counts[p] == 1]

    if chronic_proc:
        narrative += f"\nChronic procedures: {', '.join(chronic_proc)}."
    if new_proc:
        narrative += f"\nNew procedures in this visit: {', '.join(new_proc)}."
    if not chronic_proc and not new_proc:
        narrative += "\nNo procedures recorded."

    return narrative


In [28]:
def naive_narrative_prompt(row):
    narrative = f"Patient is a {row['age_at_landmark']}-year-old {row['gender']}."
    narrative += f" This is the {row['num_total_visits']} visit."
    if row['days_since_previous_visit'] != -1:
        narrative += f" The last visit happened {row['days_since_previous_visit']} days ago."
    if pd.notna(row['diag_text']) and row['diag_text'].strip():
        narrative += f" Medical history includes: {row['diag_text']}."
    if pd.notna(row['med_text']) and row['med_text'].strip():
        narrative += f" Current medications are: {row['med_text']}."
    if pd.notna(row['proc_text']) and row['proc_text'].strip():
        narrative += f" Procedures performed: {row['proc_text']}."
    # Add explicit prediction question
    narrative += " Based on this information, what is the probability of mortality within 90 days?"
    return narrative

In [27]:
print(compact_narrative_prompt(train_df_selected.iloc[1]))

You are a Doctor.
What is the probability of death in the next 90 days for 32-year-old F patient?
Visit number 2 - EMERGENCY 
Last visit happened 6.0 days ago.
DIAGNOSIS HISTORY:
Chronic diagnoses: Other ascites, Urinary tract infection, site not specified, Hyposmolality and/or hyponatremia, Acute alcoholic hepatitis, Other and unspecified alcohol dependence, continuous.
New diagnoses in this visit: Sepsis, Unspecified pleural effusion, Alcoholic fatty liver, Tobacco use disorder.
PRESCRIPTIONS HISTORY:
Chronic medications: Morphine Sulfate, Heparin, OxycoDONE (Immediate Release) , Thiamine, Multivitamins, Albumin 25% (12.5g / 50mL), Phytonadione, 0.9% Sodium Chloride, Sodium Chloride 0.9%  Flush, Magnesium Sulfate, Ondansetron, Lidocaine 5% Patch, FoLIC Acid, Nicotine Patch.
New medications in this visit: Sarna Lotion, Sodium Polystyrene Sulfonate, Lactulose, Iso-Osmotic Dextrose, CeftriaXONE, Acetylcysteine 20%, 5% Dextrose, Heparin Sodium, D5 1/2NS, Spironolactone.
PROCEDURES HISTOR

In [30]:
print(naive_narrative_prompt(train_df_selected.iloc[1]))

Patient is a 32-year-old F. This is the 2 visit. The last visit happened 6.0 days ago. Medical history includes: Alcoholic cirrhosis of liver
Pneumonia, organism unspecified
Other ascites
Portal hypertension
Urinary tract infection, site not specified
Unspecified protein-calorie malnutrition
Hyposmolality and/or hyponatremia
Other specified forms of effusion, except tuberculous
Acute alcoholic hepatitis
Hypopotassemia
Other and unspecified alcohol dependence, continuous
Other constipation
Unspecified deficiency anemia
Anxiety state, unspecified
Acute alcoholic hepatitis
Sepsis
Other ascites
Hyposmolality and/or hyponatremia
Urinary tract infection, site not specified
Unspecified pleural effusion
Alcoholic fatty liver
Other and unspecified alcohol dependence, continuous
Tobacco use disorder. Current medications are: Morphine Sulfate
Lorazepam
Heparin
Docusate Sodium
OxycoDONE (Immediate Release) 
Thiamine
Multivitamins
Ciprofloxacin HCl
Neutra-Phos
Albumin 25% (12.5g / 50mL)
Phytonadion